# BPL Bigger Dfns: a Weather Report

In [1]:
!pip install -U basedpl

In [2]:
%load_ext basedpl.notebooks

In [3]:
import basedpl
basedpl.__version__

'0.1.91'

## Building a bigger function, one piece at a time

Short dfns fit on one line. Real functions are longer: they need a plan, several steps, small helper functions, options, checks on their input, and tests. In this dialog we build one longer dfn, `report`, a step at a time, and run it after every step.

`report` takes a week of weather and gives back a record with the mean temperature, the warmest day, the wet days and the total rain. At the end, a second function, `summary`, turns that into one sentence.

BPL docs: [Dfns](https://answerdotai.github.io/basedpl/dfns.html)

Glyphs and ideas this dialog teaches:

| Glyph | Plain English |
|---|---|
| `⍝` lines as a plan | write the steps as comments first, then fill them in |
| `↑` | the first item of a list |
| `α←⍬:⍬` | a default left argument: the empty record (no options) |
| `ι⍠0⁝r` | the names (keys) in record `r` |
| `o.[k]←α.[k]` | copy the fields named `k` from `α` into `o` |
| `"message" •signal "LENGTH"` | stop with an error of your own, with a clear message |
| `⍕` | turn a number into text: `1⍕` with 1 decimal place, `⍬⍕` without the `ₓ` mark |

Ideas from earlier dialogs, used again:

| Glyph | Plain English |
|---|---|
| `{ }`, `ω`, `α` | a dfn; its right and left arguments |
| `test?answer;…` | a predicate: choose between the dfn's bodies |
| `["key":value]`, `.key` | a record, and reading one field |
| `+/÷≢` | a train: total ÷ count, the mean |
| `⍒` | positions that sort a list biggest first |
| `v.[i]` | items of `v` at positions `i` |
| `#` | keep the items where the answer is `$t` |
| `> = ≠` | compare (answers `$t` or `$f`) |
| `≢` | count the items |
| `,` | join pieces of text |
| `∞::`, `$e.message` | catch any error, and its message |
| `⍶` | the function on the left of an operator dfn |
| `◇` | separates statements on one line |

### The data

A week of midday temperatures (°C) and rainfall (mm), Monday to Sunday, as a table: a record of three columns.

In [4]:
%%bpl
days←"Mon" "Tue" "Wed" "Thu" "Fri" "Sat" "Sun"
temps←12 15 11 18 20 14 9      ⍝ °C
rain←0 2 5 0 0 1 8             ⍝ mm
week←["day":days "temp":temps "rain":rain]

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

### Plan first

#### 1. Write the steps as comments

- Before any code, write the plan inside the braces as `⍝` comments.
- The last line, `ω`, just gives the input back, so the function already runs. We'll replace it as we go.

In [5]:
%%bpl
report←{
 ⍝ 1. options
 ⍝ 2. check the input
 ⍝ 3. helpers
 ⍝ 4. the work
 ⍝ 5. the answer
 ω
}
report week       ⍝ for now: the week, unchanged

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→──────────────────────────────────────────────────────────────────────────────────────────────────┐
│ day:┌→──────────────────────────────────────────┐ temp:┌→──────────────────┐ rain:┌→────────────┐ │
│     │ "Mon" "Tue" "Wed" "Thu" "Fri" "Sat" "Sun" │      │12 15 11 18 20 14 9│      │0 2 5 0 0 1 8│ │
│     └∊──────────────────────────────────────────┘      └~──────────────────┘      └~────────────┘ │
└∊──────────────────────────────────────────────────────────────────────────────────────────────────┘

### Build it a piece at a time

#### 2. A first working version

- Start with the easy parts: the mean temperature and the total rain.
- `mean←+/÷≢` is a helper inside the dfn: total ÷ count.
- The last line builds the answer as a record. Run it and check the numbers before going on.

In [6]:
%%bpl
report←{
 mean←+/÷≢                 ⍝ helper
 avg←mean ω.temp           ⍝ the work
 total←+/ω.rain
 ["mean":avg "rain":total] ⍝ the answer
}
report week

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

["mean":14.14285714 "rain":16]

#### 3. Name each value, then build the record

- Why store `avg` first, instead of writing `["mean":mean ω.temp]`?
- **Inside square brackets, spaces separate items.** So `mean temps` inside `[ ]` is two items, the function `mean` and the list `temps`, not "mean of temps".
- The rule: work each value out into a name, then build the record from the names.

In [7]:
%%bpl
mean←+/÷≢
["mean":mean temps]   ⍝ not what we want: mean and temps as separate items
avg←mean temps
["mean":avg]          ⍝ right: the value first, then the record

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→──────────────────────────────────┐
│ mean:⟨+/÷≢⟩ ┌→──────────────────┐ │
│             │12 15 11 18 20 14 9│ │
│             └~──────────────────┘ │
└∊──────────────────────────────────┘

["mean":14.14285714]

#### 4. The warmest day

- `⍒temps` gives the positions, hottest first. Friday (position 4) comes first.
- `↑` takes the **first** item of a list, so `↑⍒temps` is 4.
- `days.[4]` is `"Fri"`.

In [8]:
%%bpl
temps
⍒temps               ⍝ positions, hottest first
↑⍒temps              ⍝ just the first: 4
days.[↑⍒temps]       ⍝ "Fri"

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

12 15 11 18 20 14 9

[4 3 1 5 0 2 6]ₓ

4ₓ

"Fri"

#### 5. The wet days, and the next version

- `rain>0` marks the wet days, and `#` keeps those days' names.
- Add both new lines to `report`. It's still short: each line does one thing.

In [9]:
%%bpl
rain
rain>0
(rain>0)#days        ⍝ the wet days

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

0 2 5 0 0 1 8

$f $t $t $f $f $t $t

┌→────────────────────────┐
│ "Tue" "Wed" "Sat" "Sun" │
└∊────────────────────────┘

In [10]:
%%bpl
report←{
 mean←+/÷≢
 avg←mean ω.temp
 total←+/ω.rain
 warmest←ω.day.[↑⍒ω.temp]
 wet←(ω.rain>0)#ω.day
 ["mean":avg "warmest":warmest "wet":wet "rain":total]
}
report week

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→───────────────────────────────────────────────────────────────────────┐
│ mean:14.14285714 warmest:"Fri" wet:┌→────────────────────────┐ rain:16 │
│                                    │ "Tue" "Wed" "Sat" "Sun" │         │
│                                    └∊────────────────────────┘         │
└∊───────────────────────────────────────────────────────────────────────┘

### Make it flexible

#### 6. A helper for a choice

- Next, an option to report in °F. The function needs a choice: convert or not.
- A predicate's `;` separates the bodies of the **whole** dfn. A choice written in the middle of `report` would cut `report` in two.
- So give the choice its own small dfn, `conv`. Here `unit` is a name outside `conv`, and `conv` can read it.

In [11]:
%%bpl
unit←"F"
conv←{unit≡"F"?32+1.8×ω;ω}   ⍝ to °F, or leave as it is
conv 0 100                   ⍝ 32 212
unit←"C"
conv 0 100                   ⍝ 32 212

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

32 212

0 100

#### 7. Options with defaults

`report` now accepts an options record as its left argument:

```bpl
report week                 ⍝ no options
["unit":"F"] report week    ⍝ override one option
```

##### Default left argument

```bpl
α←⍬:⍬
```

Read this from right to left:

- the right `⍬` is an empty vector;
- `⍬:⍬` gives it no keys, making an empty record;
- `α←` gives `α` that value **only when the caller omitted `α`**.

Thus `α` is always a record: either the caller's options or an empty one.

##### Merge options over defaults

Suppose `α` is `["unit":"F"]`.

**1. Make the defaults:**

```bpl
o←["unit":"C" "wet":0]
```

- `o` is a new local name.
- `←` assigns the value on its right to the name on its left.
- `[ … ]` builds a record here because its items use `key:value`.
- `"unit":"C"` and `"wet":0` are the two keyed fields.

So `o` starts as `["unit":"C" "wet":0]`.

**2. Get the caller's keys:**

```bpl
k←ι⍠0⁝α
```

Read the right side as `ι⍠0 α`:

- `α` is the caller's options record.
- `ι` means **indices**; on a keyed axis it returns the keys.
- `⍠0` tells `ι` to use axis `0`—the record's keyed axis.
- `⁝` only separates tokens, so `0` and `α` do not join into one strand.
- `k←` stores the result in `k`.

Here `k` becomes `"unit"`. With no options, it is empty.

**3. Copy those keyed fields:**

```bpl
o.[k]←α.[k]
```

Read right to left:

- `.` introduces field/index selection.
- `[k]` supplies the key or keys to select.
- `α.[k]` reads those fields from the caller's record: here, `"F"`.
- `o.[k]←` assigns that value to the same fields in `o`.

The result is `["unit":"F" "wet":0]`: `unit` was replaced; `wet` kept its default.

##### Use the completed options

```bpl
conv←{o.unit≡"F"?32+1.8×ω;ω}
wet←(ω.rain>o.wet)#ω.day
```

`>` is strict: with `"wet":2`, exactly 2 mm is not counted as wet.

In [12]:
%%bpl
ι⍠0⁝["unit":"F" "wet":1]     ⍝ the names in a record

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→─────────────┐
│ "unit" "wet" │
└∊─────────────┘

In [13]:
%%bpl
report←{
 ⍝ 1. options: defaults, then any the caller gave
 α←⍬:⍬
 o←["unit":"C" "wet":0]
 k←ι⍠0⁝α
 o.[k]←α.[k]
 ⍝ 3. helpers
 mean←+/÷≢
 conv←{o.unit≡"F"?32+1.8×ω;ω}
 ⍝ 4. the work
 t←conv ω.temp
 avg←mean t
 total←+/ω.rain
 warmest←ω.day.[↑⍒t]
 wet←(ω.rain>o.wet)#ω.day
 ⍝ 5. the answer
 ["mean":avg "warmest":warmest "wet":wet "rain":total]
}
report week                ⍝ defaults
["unit":"F"] report week   ⍝ in °F
["wet":1] report week      ⍝ only days with more than 1 mm

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→───────────────────────────────────────────────────────────────────────┐
│ mean:14.14285714 warmest:"Fri" wet:┌→────────────────────────┐ rain:16 │
│                                    │ "Tue" "Wed" "Sat" "Sun" │         │
│                                    └∊────────────────────────┘         │
└∊───────────────────────────────────────────────────────────────────────┘

┌→───────────────────────────────────────────────────────────────────────┐
│ mean:57.45714286 warmest:"Fri" wet:┌→────────────────────────┐ rain:16 │
│                                    │ "Tue" "Wed" "Sat" "Sun" │         │
│                                    └∊────────────────────────┘         │
└∊───────────────────────────────────────────────────────────────────────┘

┌→─────────────────────────────────────────────────────────────────┐
│ mean:14.14285714 warmest:"Fri" wet:┌→──────────────────┐ rain:16 │
│                                    │ "Tue" "Wed" "Sun" │         │
│                                    └∊──────────────────┘         │
└∊─────────────────────────────────────────────────────────────────┘

### Make it safe

#### 8. Check the input

- Weeks with a different number of temperatures and rain values make no sense. Stop with a clear message rather than a confusing answer.
- `"…" •signal "LENGTH"` stops the function with your own message, as a `LENGTH` error.
- The check line ends with `;`: if the test is true, the function stops there. Otherwise it carries on into the next body. Names made before the `;` (like `o`) are still there.
- `why` (from the dfns dialog) catches the error and shows its message.

In [14]:
%%bpl
report←{
 ⍝ 1. options: defaults, then any the caller gave
 α←⍬:⍬
 o←["unit":"C" "wet":0]
 k←ι⍠0⁝α
 o.[k]←α.[k]
 ⍝ 2. check the input: stop with a clear message
 (≢ω.temp)≠≢ω.rain?"temp and rain need one value per day" •signal "LENGTH"
 ;
 ⍝ 3. helpers
 mean←+/÷≢
 conv←{o.unit≡"F"?32+1.8×ω;ω}
 ⍝ 4. the work
 t←conv ω.temp
 avg←mean t
 total←+/ω.rain
 warmest←ω.day.[↑⍒t]
 wet←(ω.rain>o.wet)#ω.day
 ⍝ 5. the answer
 ["mean":avg "warmest":warmest "wet":wet "rain":total]
}
report week                    ⍝ still works

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→───────────────────────────────────────────────────────────────────────┐
│ mean:14.14285714 warmest:"Fri" wet:┌→────────────────────────┐ rain:16 │
│                                    │ "Tue" "Wed" "Sat" "Sun" │         │
│                                    └∊────────────────────────┘         │
└∊───────────────────────────────────────────────────────────────────────┘

In [15]:
%%bpl
bad←["day":days "temp":temps "rain":1 2]   ⍝ only 2 rain values
why←{∞::$e.message ◇ ⍶ ω}                   ⍝ show the error message
report why bad

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

"temp and rain need one value per day"

#### Why use `why`?

`report` already checks its input and triggers the error itself. With the bad data:

```bpl
report bad
```

it reaches `•signal`, stops the calculation and reports a `LENGTH ERROR`. Because that error is uncaught, it also stops the rest of the cell and the notebook shows a traceback.

`why` does **not** create the error. It is a small wrapper for demonstrating an error without stopping the cell:

```bpl
why←{∞::$e.message ◇ ⍶ ω}
report why bad
```

Read its parts like this:

- `⍶` is the function on the left of `why` - here, `report`.
- `⍶ ω` runs `report bad` normally.
- `∞::` catches any error from that call.
- `$e.message` gives back the error's message as an ordinary text value.

So the two calls serve different purposes:

- `report bad` shows the real, uncaught error and stops.
- `report why bad` catches the same error and returns `"temp and rain need one value per day"`, which is convenient for examples and tests.

In normal use, call `report` directly. Use `why` only when you deliberately want to inspect an expected error without interrupting the surrounding calculation.

### Test and tidy

#### 9. Test with a tiny week
These tests check that `report` still gives the correct answers after we change or extend it.

- Make a small example with only three days, called A, B, and C. Calculate the correct answers yourself first, then check that `report` gives the same answers.
- Work out the answers first: `(10+20+15)÷3` is 15, B is warmest, B is the only wet day, and the total rain is 3.
- Test the option path too: converting the mean of 15 °C should give 59 °F.
- Each comparison below is a small regression test and should return `$t`. If a later change breaks one, it shows which behaviour changed.
- We can run the tests again after every change to `report`.

In [16]:
%%bpl
d←"A" "B" "C"
t←10 20 15
r←0 3 0
tiny←["day":d "temp":t "rain":r]
report tiny
res←report tiny
res.mean=15
res.warmest≡"B"
res.wet≡,⊂"B"          ⍝ a list holding the one day "B"
res.rain=3
(["unit":"F"] report tiny).mean=59

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

┌→───────────────────────────────────────┐
│ mean:15 warmest:"B" wet:┌→────┐ rain:3 │
│                         │ "B" │        │
│                         └∊────┘        │
└∊───────────────────────────────────────┘

$t

$t

$t

$t

$t

#### 10. Build on it: `summary`

- A second, short function can use `report`. The wrapper function `summary` passes its options straight on (`α report ω`), then builds a sentence.
- `⍕` turns numbers into text: `1⍕` rounds to 1 decimal place, and `⍬⍕` writes a count without the `ₓ` mark. `,` joins the pieces.

In [17]:
%%bpl
summary←{
 r←α report ω
 n←≢r.wet
 "Warmest: ",r.warmest,". Wet days: ",(⍬⍕n),". Mean: ",1⍕r.mean
}
summary week
["unit":"F"] summary week

Javascript(if (window.loadBplInput) window.loadBplInput();
else import('https://cdn.jsdelivr.net/npm/basedpl/browser/bpl-editor.js').then(m => m.install());)

"Warmest: Fri. Wet days: 4. Mean: 14.1"

"Warmest: Fri. Wet days: 4. Mean: 57.5"

The finished `report` has five clear parts, each marked by its comment: options, a check, helpers, the work, and the answer. Each line does one thing, and the tiny-week tests tell you at once if a change breaks something.

### Your turn

1. Add a `dry` field to the answer: the number of days with no rain.
   *Hint:* `+/ω.rain=0`, stored in a name first.
2. Add a `hot` option (default 18) and a `hot` field listing the days above it.
   *Hint:* add `"hot":18` to the defaults, then `(ω.temp>o.hot)#ω.day`.
3. Add a second check: `day` must also have one name per temperature.
   *Hint:* another test line ending in `;`.
4. Add the total rain to `summary`'s sentence.
5. Make a week of your own and run `report` and `summary` on it. Add a tiny-week test for each new field.